In [ ]:
import os
import shutil

import numpy as np
import pandas as pd
import xarray as xr
from cdo import Cdo
from joblib import Parallel, delayed

In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
# --- rasterize pcolormesh/pcolor in vector output (PDF/SVG) so maps match the PNG (no inter-cell seams) ---
import matplotlib.axes as _mpl_axes_raster
for _pcm_name in ("pcolormesh", "pcolor"):
    _pcm_orig = getattr(_mpl_axes_raster.Axes, _pcm_name)
    if not getattr(_pcm_orig, "_rasterized_patch", False):
        def _pcm_make(_orig):
            def _wrapped(self, *a, **k):
                k.setdefault("rasterized", True)
                _m = _orig(self, *a, **k)
                try: _m.set_edgecolor("face")
                except Exception: pass
                return _m
            _wrapped._rasterized_patch = True
            return _wrapped
        setattr(_mpl_axes_raster.Axes, _pcm_name, _pcm_make(_pcm_orig))
plt.rcParams["savefig.dpi"] = 200   # crisp rasterized meshes in the PDF (PNG keeps its explicit dpi)
# --- end rasterize patch ---
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
from shapely.geometry import Point
from shapely.ops import unary_union
from shapely.prepared import prep
try:
    from shapely import contains_xy
except ImportError:
    contains_xy = None


REGION_NAME = "western_europe"
EVENT_START_DATE = "2026-05-15"
EVENT_END_DATE = "2026-06-01"
EVENT_NAME = f"era5t_{EVENT_START_DATE}_to_{EVENT_END_DATE}"

TARGET_DAY = "2026-05-25"
CLIM_START_YEAR = 1995
CLIM_END_YEAR = 2025
CLIM_HALF_WINDOW_DAYS = 3

START_YEAR = 1940
END_YEAR = 2025
MONTH_NAME = "May"

G0 = 9.80665
Z500_REF_GEOPOTENTIAL = 5863 * 9.81

MAP_LON_MIN, MAP_LON_MAX = -15, 15
MAP_LAT_MIN, MAP_LAT_MAX = 35, 60

LON_MIN, LON_MAX = -5.5, 5
LAT_MIN, LAT_MAX = 44.5, 55

PARIS_LAT, PARIS_LON = 48.85, 2.35

FS_PANEL = 13
FS_LABEL = 13
FS_TICK = 12
FS_TITLE = 13
FS_CLABEL = 11


# ---------------------------------------------------------------------------
# Analogue-consistent Z500 detrend (same RMST + slope as the analogue notebook)
# ---------------------------------------------------------------------------
DETREND_Z500 = False         # panel (a) z500 contours = ABSOLUTE (raw), not detrended
DETREND_Z500_VIOLIN = True   # detrend the panel (b) z500 violin

ANALOGUE_RAWDAT = (
    "${ANALOGUE_BASE}/rawdat/"
)
ANALOGUE_Z500_FILE = os.path.join(
    ANALOGUE_RAWDAT, "era5_z500_daily_NA_1940_2026_FULL_1deg.nc"
)

# 0.5-degree ERA5 daily-mean t2m: the single anomaly reference shared by
# panels (a), (e) and (f) so the map and the IFS composites are consistent.
RAWDAT_0P5 = (
    "${ANALOGUE_BASE}/rawdat_0p5/"
)
ERA5_T2M_0P5_FILE = os.path.join(
    RAWDAT_0P5, "era5_t2m_daily_NA_1940_2026_FULL_0p5deg.nc"
)
AN_LAT_MIN, AN_LAT_MAX = 30, 68
AN_LON_MIN, AN_LON_MAX = -20, 25
AN_MONTH_MIN, AN_MONTH_MAX = 4, 8
RMST_LAT_MIN, RMST_LAT_MAX = 35, 70
RMST_LON_MIN, RMST_LON_MAX = -15, 30
RMST_T2M_FILE = ERA5_T2M_0P5_FILE   # same 0.5-deg daily t2m as the Fig. 5 analogue notebook's RMST
RMST_ROLL_YEARS = 15                # centred rolling-mean length [years]
RMST_EXTRAP_YEARS = 30              # years of smoothed RMST used for the linear edge extrapolation


IPCC_TEMP_DIV_FILE = "temp_div.txt"
IPCC_TEMP_DIV_URLS = [
    "https://raw.githubusercontent.com/IPCC-WG1/colormaps/master/"
    "continuous_colormaps_rgb_0-1/temp_div.txt",
    "https://raw.githubusercontent.com/IPCC-WG1/colormaps/main/"
    "continuous_colormaps_rgb_0-1/temp_div.txt",
]


def ipcc_temp_div_cmap():
    if not os.path.exists(IPCC_TEMP_DIV_FILE):
        for url in IPCC_TEMP_DIV_URLS:
            try:
                urllib.request.urlretrieve(url, IPCC_TEMP_DIV_FILE)
                print(f"[INFO] Downloaded IPCC temp_div colormap from {url}")
                break
            except Exception:
                continue
    if os.path.exists(IPCC_TEMP_DIV_FILE):
        rgb = np.loadtxt(IPCC_TEMP_DIV_FILE)
        return mcolors.ListedColormap(rgb, name="ipcc_temp_div")
    print("[WARN] temp_div.txt not found and download failed; using RdBu_r")
    return plt.get_cmap("RdBu_r")


def anomaly_discrete_scale(limit, step):
    """Discrete diverging T-anomaly scale: `step`-degC bins symmetric about 0,
    with the two bins straddling 0 forced white. Returns (norm, cmap, levels)."""
    levels = np.arange(-limit, limit + step, step)
    ncolors = len(levels) - 1
    colors = ipcc_temp_div_cmap()(np.linspace(0.0, 1.0, ncolors))
    mid = ncolors // 2                     # bins on either side of 0 -> white
    colors[mid - 1] = mcolors.to_rgba("white")
    colors[mid] = mcolors.to_rgba("white")
    cmap = mcolors.ListedColormap(colors)
    norm = mcolors.BoundaryNorm(levels, cmap.N)
    return norm, cmap, levels


COMPILED_FILE = (
    f"${WORK}/02_MSE/{REGION_NAME}/"
    f"era5t_events/{EVENT_NAME}/era5t_{REGION_NAME}_{EVENT_START_DATE}_to_"
    f"{EVENT_END_DATE}_compiled.nc"
)
HOURLY_MSE_FILE = (
    f"${WORK}/02_MSE/{REGION_NAME}/"
    f"era5t_events/{EVENT_NAME}/era5t_{REGION_NAME}_{EVENT_START_DATE}_to_"
    f"{EVENT_END_DATE}_hourly_mse.nc"
)
HIST_DAILY_TASMAX_FILE = (
    f"${WORK}/02_MSE/{REGION_NAME}/"
    f"era5_{REGION_NAME}_{MONTH_NAME}_{START_YEAR}_{END_YEAR}_compiled.nc"
)
HIST_DAILY_TASMAX_VAR = "tasmax"

HIST_DIR = f"${WORK}/02_MSE/{REGION_NAME}/annual_may_gridcell"
HIST_EXTREMES_FILE = os.path.join(
    HIST_DIR,
    f"era5_{REGION_NAME}_{MONTH_NAME}_{START_YEAR}_{END_YEAR}"
    "_annual_gridcell_may_extremes.nc",
)

OUT_PNG = os.path.join(
    os.getcwd(),
    f"Figure1_{EVENT_NAME}_{TARGET_DAY}.pdf",
)


def require_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required input file: {path}")


def coord_name(obj, candidates=("lat", "latitude")):
    for name in candidates:
        if name in obj.coords:
            return name
    raise KeyError(f"None of these coordinates were found: {candidates}")


def add_panel_label(ax, label):
    ax.text(-0.08, 1.04, label, transform=ax.transAxes, fontsize=FS_PANEL,
            fontweight="bold", va="bottom", ha="left")


def to_pm180(da):
    lon_name = coord_name(da, ("lon", "longitude"))
    if float(da[lon_name].max()) > 180:
        da = da.assign_coords(
            {lon_name: (((da[lon_name] + 180) % 360) - 180)}
        ).sortby(lon_name)
    return da


def regrid_nearest(da, ref):
    ref_lat = coord_name(ref, ("lat", "latitude"))
    ref_lon = coord_name(ref, ("lon", "longitude"))
    da_lat = coord_name(da, ("lat", "latitude"))
    da_lon = coord_name(da, ("lon", "longitude"))
    rename_coords = {}
    if da_lat != ref_lat:
        rename_coords[da_lat] = ref_lat
    if da_lon != ref_lon:
        rename_coords[da_lon] = ref_lon
    if rename_coords:
        da = da.rename(rename_coords)
    same_grid = (
        da.sizes.get(ref_lat) == ref.sizes[ref_lat]
        and da.sizes.get(ref_lon) == ref.sizes[ref_lon]
        and np.allclose(da[ref_lat].values, ref[ref_lat].values)
        and np.allclose(da[ref_lon].values, ref[ref_lon].values)
    )
    if same_grid:
        return da.assign_coords(
            {ref_lat: ref[ref_lat].values, ref_lon: ref[ref_lon].values}
        )
    return da.interp({ref_lat: ref[ref_lat], ref_lon: ref[ref_lon]}, method="nearest")


def clipped_map_extent(da):
    lat_name = coord_name(da, ("lat", "latitude"))
    lon_name = coord_name(da, ("lon", "longitude"))
    lon_lo = max(MAP_LON_MIN, float(da[lon_name].min()))
    lon_hi = min(MAP_LON_MAX, float(da[lon_name].max()))
    lat_lo = max(MAP_LAT_MIN, float(da[lat_name].min()))
    lat_hi = min(MAP_LAT_MAX, float(da[lat_name].max()))
    if lon_hi <= lon_lo or lat_hi <= lat_lo:
        raise ValueError(
            "Preferred map window does not overlap the event file coverage; "
            "adjust MAP_LON/LAT bounds."
        )
    if (lon_lo, lon_hi, lat_lo, lat_hi) != (
        MAP_LON_MIN, MAP_LON_MAX, MAP_LAT_MIN, MAP_LAT_MAX
    ):
        print(f"[WARN] Map extent clipped to data coverage: "
              f"lon {lon_lo:.1f}..{lon_hi:.1f}, lat {lat_lo:.1f}..{lat_hi:.1f}")
    return [lon_lo, lon_hi, lat_lo, lat_hi]


def _sel_lat_range(da, lat_name, lo, hi):
    lat = da[lat_name]
    if bool(lat[0] > lat[-1]):
        return da.sel({lat_name: slice(hi, lo)})
    return da.sel({lat_name: slice(lo, hi)})


def _analogue_rmst():
    """Daily European RMST, exactly as the analogue (Fig. 5) notebook's load_rmst_full():
    cos(lat)-weighted mean of daily T2M over RMST_LAT/LON, 15-yr centred running mean;
    the NaN edges left by the window are filled with a linear trend fitted to the
    first / last RMST_EXTRAP_YEARS of the smoothed series (continuous at the joins)."""
    with xr.open_dataarray(RMST_T2M_FILE) as src:
        da = to_pm180(src.squeeze(drop=True))
        lat_name = coord_name(da, ("lat", "latitude"))
        lon_name = coord_name(da, ("lon", "longitude"))
        da = _sel_lat_range(da, lat_name, RMST_LAT_MIN, RMST_LAT_MAX)
        da = da.sel({lon_name: slice(RMST_LON_MIN, RMST_LON_MAX)})
        weights = np.cos(np.deg2rad(da[lat_name]))
        rmst = da.weighted(weights).mean(dim=(lat_name, lon_name)).load()
    if float(rmst.mean()) > 150:                                   # K -> degC
        rmst = rmst - 273.15
    rmst["time"] = rmst.indexes["time"].normalize()

    # 15-yr centred running mean (full window required -> NaN at both edges)
    rmst = rmst.rolling(time=int(round(RMST_ROLL_YEARS * 365.25)), center=True).mean()
    valid = rmst.dropna("time")
    n_fit = int(round(RMST_EXTRAP_YEARS * 365.25))
    t_head = rmst["time"] < valid["time"][0]
    t_tail = rmst["time"] > valid["time"][-1]

    def _edge_trend(fit_part, anchor, times):
        coef = fit_part.polyfit(dim="time", deg=1).polyfit_coefficients
        line = xr.polyval(times, coef)
        return line - (float(xr.polyval(anchor["time"], coef)) - float(anchor))   # pass through anchor

    trend_past = _edge_trend(valid[:n_fit], valid[0], rmst["time"][t_head])
    trend_future = _edge_trend(valid[-n_fit:], valid[-1], rmst["time"][t_tail])
    print(f"[INFO] RMST: {RMST_ROLL_YEARS}-yr centred mean; filled {int(t_head.sum())} leading / "
          f"{int(t_tail.sum())} trailing days by {RMST_EXTRAP_YEARS}-yr linear extrapolation")
    return rmst.fillna(trend_past).fillna(trend_future).rename("rmst")


# Cache the RMST series so panels (a) and (b) share one computation.
_RMST_CACHE = {}


def analogue_rmst_cached():
    if "rmst" not in _RMST_CACHE:
        require_file(RMST_T2M_FILE)
        _RMST_CACHE["rmst"] = _analogue_rmst()
    return _RMST_CACHE["rmst"]


def _rmst_at_year(rmst, year, month_day="05-25"):
    return float(
        rmst.sel(time=np.datetime64(f"{int(year):04d}-{month_day}"), method="nearest")
    )


def _series_years(series):
    """Best-effort integer years for a per-year historical series."""
    if "time" in series.coords:
        vals = series["time"].values
        try:
            return pd.DatetimeIndex(vals).year.values
        except (TypeError, ValueError):
            return np.asarray(vals).astype(int)
    if "year" in series.coords:
        return np.asarray(series["year"].values).astype(int)
    return np.asarray(series[series.dims[0]].values).astype(int)


def z500_detrend_offset():
    """Scalar (m2 s-2) to subtract from the event Z500 in panel (a)."""
    require_file(ANALOGUE_Z500_FILE)
    rmst = analogue_rmst_cached()
    with xr.open_dataarray(ANALOGUE_Z500_FILE) as src:
        lat_name = coord_name(src, ("lat", "latitude"))
        lon_name = coord_name(src, ("lon", "longitude"))
        z = _sel_lat_range(src, lat_name, AN_LAT_MIN, AN_LAT_MAX)
        z = z.sel({lon_name: slice(AN_LON_MIN, AN_LON_MAX)})
        z = z.where(
            (z["time.month"] >= AN_MONTH_MIN) & (z["time.month"] <= AN_MONTH_MAX),
            drop=True,
        ).squeeze(drop=True).load()
        weights = np.cos(np.deg2rad(z[lat_name]))
        z_box = z.weighted(weights).mean(dim=(lat_name, lon_name))
    rmst_amjja = rmst.where(
        (rmst["time.month"] >= AN_MONTH_MIN) & (rmst["time.month"] <= AN_MONTH_MAX),
        drop=True,
    )
    z_box, rmst_amjja = xr.align(z_box, rmst_amjja, join="inner")
    slope = float(np.polyfit(rmst_amjja.values, z_box.values, 1)[0])
    mean_rmst = float(rmst_amjja.mean())
    rmst_event = float(
        rmst_amjja.sel(
            time=np.datetime64(pd.Timestamp(TARGET_DAY).normalize()), method="nearest"
        )
    )
    offset = (rmst_event - mean_rmst) * slope
    print(f"[INFO] Z500 detrend (analogue RMST): slope={slope:.1f} m2s-2/K, "
          f"RMST_event={rmst_event:.3f}, mean_RMST={mean_rmst:.3f}, "
          f"offset={offset:.1f} m2s-2 ({offset / G0:.1f} m)")
    return offset


def land_geometry():
    land_shp = shapereader.natural_earth(
        resolution="10m", category="physical", name="land"
    )
    geometries = list(shapereader.Reader(land_shp).geometries())
    return unary_union(geometries)


LAND_GEOM = land_geometry()
PREPARED_LAND_GEOM = prep(LAND_GEOM)


def land_mask_for_grid(lat, lon):
    lon_2d, lat_2d = np.meshgrid(lon.values, lat.values)
    lon_for_mask = ((lon_2d + 180) % 360) - 180
    if contains_xy is not None:
        mask = contains_xy(LAND_GEOM, lon_for_mask, lat_2d)
    else:
        flat_mask = [
            PREPARED_LAND_GEOM.contains(Point(x, y))
            for x, y in zip(lon_for_mask.ravel(), lat_2d.ravel())
        ]
        mask = np.asarray(flat_mask, dtype=bool).reshape(lon_2d.shape)
    return xr.DataArray(
        mask, coords={lat.name: lat, lon.name: lon},
        dims=(lat.name, lon.name), name="land_mask",
    )


def big_box_land_mean(da):
    lat_name = coord_name(da, ("lat", "latitude"))
    lon_name = coord_name(da, ("lon", "longitude"))
    lat = da[lat_name]
    lon = da[lon_name]
    lat_mask = (lat >= LAT_MIN) & (lat <= LAT_MAX)
    if float(lon.max()) > 180:
        lon_min = LON_MIN % 360
        lon_max = LON_MAX % 360
        if lon_min <= lon_max:
            lon_mask = (lon >= lon_min) & (lon <= lon_max)
        else:
            lon_mask = (lon >= lon_min) | (lon <= lon_max)
    else:
        lon_mask = (lon >= LON_MIN) & (lon <= LON_MAX)
    da_box = da.where(lat_mask & lon_mask, drop=True)
    if da_box.sizes.get(lat_name, 0) == 0 or da_box.sizes.get(lon_name, 0) == 0:
        raise ValueError("Big-region box selection is empty.")
    land_mask = land_mask_for_grid(da_box[lat_name], da_box[lon_name])
    da_land = da_box.where(land_mask)
    return da_land.mean(dim=(lat_name, lon_name), skipna=True).load()


def tasmax_running_window_climatology(ref):
    require_file(HIST_DAILY_TASMAX_FILE)
    target_doy = pd.Timestamp(TARGET_DAY).dayofyear
    with xr.open_dataset(HIST_DAILY_TASMAX_FILE) as ds:
        if HIST_DAILY_TASMAX_VAR not in ds:
            raise KeyError(
                f"Expected '{HIST_DAILY_TASMAX_VAR}' in {HIST_DAILY_TASMAX_FILE}, "
                f"found: {list(ds.data_vars)}"
            )
        da = ds[HIST_DAILY_TASMAX_VAR]
        da = da.sel(time=slice(f"{CLIM_START_YEAR}-01-01", f"{CLIM_END_YEAR}-12-31"))
        if da.sizes.get("time", 0) == 0:
            raise ValueError(
                f"No historical tasmax timesteps in {CLIM_START_YEAR}-{CLIM_END_YEAR}."
            )
        doy = da["time"].dt.dayofyear
        doy_dist = np.abs(((doy - target_doy + 182) % 365) - 182)
        da_window = da.sel(time=doy_dist <= CLIM_HALF_WINDOW_DAYS)
        if da_window.sizes.get("time", 0) == 0:
            raise ValueError(
                f"No historical tasmax days within +-{CLIM_HALF_WINDOW_DAYS} "
                f"days of {TARGET_DAY}."
            )
        n_days = int(da_window.sizes["time"])
        clim = da_window.mean("time").load()
    print(f"[INFO] Climatology from {n_days} historical days "
          f"(window +-{CLIM_HALF_WINDOW_DAYS} d around {TARGET_DAY})")
    clim = to_pm180(clim)
    return regrid_nearest(clim, ref)


_ERA5_T2M_0P5_CACHE = {}


def era5_t2m_0p5_dataarray():
    """ERA5 0.5-deg daily-mean t2m in degC (full record incl. 2026 from ERA5T)."""
    if "da" not in _ERA5_T2M_0P5_CACHE:
        require_file(ERA5_T2M_0P5_FILE)
        da = to_pm180(xr.open_dataarray(ERA5_T2M_0P5_FILE).squeeze(drop=True).load())
        if float(da.mean()) > 100:                       # K -> degC
            da = da - 273.15
        _ERA5_T2M_0P5_CACHE["da"] = da
    return _ERA5_T2M_0P5_CACHE["da"]


def era5_t2m_0p5_climatology():
    """Daily-mean t2m climatology (degC) for TARGET_DAY's calendar day:
    CLIM_START_YEAR..CLIM_END_YEAR, +-CLIM_HALF_WINDOW_DAYS around that day.
    Same window definition as the composite reference, on the ERA5 0.5-deg grid."""
    if "clim" not in _ERA5_T2M_0P5_CACHE:
        da = era5_t2m_0p5_dataarray()
        t = pd.DatetimeIndex(da["time"].values)
        target_doy = pd.Timestamp(TARGET_DAY).dayofyear
        doy = t.dayofyear.values
        doy_dist = np.abs(((doy - target_doy + 182) % 365) - 182)
        in_win = ((doy_dist <= CLIM_HALF_WINDOW_DAYS)
                  & (t.year >= CLIM_START_YEAR) & (t.year <= CLIM_END_YEAR))
        if not in_win.any():
            raise ValueError("No ERA5 0.5-deg t2m days in the climatology window.")
        clim = da.isel(time=np.where(in_win)[0]).mean("time").load()
        print(f"[INFO] ERA5 0.5deg t2m clim from {int(in_win.sum())} days "
              f"({CLIM_START_YEAR}-{CLIM_END_YEAR}, +-{CLIM_HALF_WINDOW_DAYS} d "
              f"around {TARGET_DAY})")
        _ERA5_T2M_0P5_CACHE["clim"] = clim
    return _ERA5_T2M_0P5_CACHE["clim"]


def plot_panel_map(fig, ax):
    # Panel (a): ERA5 0.5-deg daily-mean T2m anomaly on TARGET_DAY -- same field,
    # colormap and pcolormesh as the IFS composites (e/f) -- with daily-mean
    # z500 height contours from the event file overlaid.
    event_t2m = era5_t2m_0p5_dataarray().sel(
        time=pd.Timestamp(TARGET_DAY), method="nearest"
    )
    anomaly = event_t2m - era5_t2m_0p5_climatology()

    lat_name = coord_name(anomaly, ("lat", "latitude"))
    lon_name = coord_name(anomaly, ("lon", "longitude"))

    ax.set_extent([COMP_MAP_LON_MIN, COMP_MAP_LON_MAX, COMP_MAP_LAT_MIN, COMP_MAP_LAT_MAX],
              crs=ccrs.PlateCarree())
    ax.coastlines(resolution="10m", linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4)
    ax.add_feature(cfeature.OCEAN, alpha=0.15)

    # Identical discrete colour scale + pcolormesh as panels (e)/(f):
    # 1-degC steps, symmetric about 0, with 0 sitting on a white bin.
    anom_norm, anom_cmap, anom_levels = anomaly_discrete_scale(
        COMP_T_ANOM_LIMIT, COMP_T_ANOM_STEP
    )
    mesh = ax.pcolormesh(
        anomaly[lon_name], anomaly[lat_name], anomaly,
        cmap=anom_cmap, norm=anom_norm, shading="auto",
        transform=ccrs.PlateCarree(),
    )
    cb = fig.colorbar(mesh, ax=ax, orientation="vertical", shrink=0.85, pad=0.02,
                      ticks=anom_levels[::2])
    cb.set_label("ERA5 Daily Mean 2m Temp. Anomaly [°C]", fontsize=FS_LABEL)
    cb.ax.tick_params(labelsize=FS_TICK)

    # Daily-mean z500 as geopotential height contours [m], optionally detrended
    # (same RMST as the analogues): a spatially-uniform shift, pattern unchanged.
    require_file(COMPILED_FILE)
    with xr.open_dataset(COMPILED_FILE) as ds:
        if "z_mean" not in ds:
            raise KeyError(
                "Compiled event file has no 'z_mean'. Re-run the extraction "
                "script with the daily_mean aggregation for z and recompile."
            )
        ds_day = ds.sel(time=TARGET_DAY)
        if "time" in ds_day.dims:
            ds_day = ds_day.squeeze("time", drop=True)
        z_mean_day = to_pm180(ds_day["z_mean"]).load()

    z_offset = z500_detrend_offset() if DETREND_Z500 else 0.0
    z_height = (z_mean_day - z_offset) / G0
    z_lat = coord_name(z_height, ("lat", "latitude"))
    z_lon = coord_name(z_height, ("lon", "longitude"))
    contours = ax.contour(
        z_height[z_lon], z_height[z_lat], z_height,
        levels=np.arange(4800, 6000, 40), colors="black", linewidths=0.9,
        transform=ccrs.PlateCarree(),
    )
    ax.clabel(contours, inline=True, fontsize=FS_CLABEL, fmt="%d")

    ax.plot(
        [LON_MIN, LON_MAX, LON_MAX, LON_MIN, LON_MIN],
        [LAT_MIN, LAT_MIN, LAT_MAX, LAT_MAX, LAT_MIN],
        color="black", linestyle="--", linewidth=1.8,
        transform=ccrs.PlateCarree(), zorder=6,
    )
    ax.plot(
        PARIS_LON, PARIS_LAT, marker="x", color="black", markersize=9,
        markeredgewidth=2.2, transform=ccrs.PlateCarree(), zorder=7,
    )
    gl = ax.gridlines(draw_labels=True, linestyle="--", alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": FS_TICK}
    gl.ylabel_style = {"size": FS_TICK}
        # Box-mean summary legend (same box + box-mean estimator as panels e/f).
    box_t2m_anom = float(comp_target_box_mean(anomaly, lon_name, lat_name))
    box_z500 = float(comp_target_box_mean(z_height, z_lon, z_lat))
    ax.legend(
        handles=[
            Line2D([0], [0], color="none",
                   label=f"box T2m anom: {box_t2m_anom:+.2f} °C"),
            Line2D([0], [0], color="none",
                   label=f"box Z500: {box_z500:.1f} m"),
        ],
        loc="lower right", title=f"ERA5 {pd.Timestamp(TARGET_DAY):%d %b %Y}",
        framealpha=0.9, fontsize=FS_TICK - 1, title_fontsize=FS_TICK - 1,
        handlelength=0, handletextpad=0,
    )
    add_panel_label(ax, "(a)")
    add_panel_label(ax, "(a)")


VIOLIN_SPECS = [
    {
        "label": "May TXx", "units": "$^\\circ$C", "hist_var": "tasmax_may_max",
        "event_source": "tasmax", "mode": "max", "scale": 1.0, "offset": -273.15,
    },
    {
        "label": "Detrended May Z500 max", "units": "m", "hist_var": "z_may_max",
        "event_source": "z_max", "mode": "max", "scale": 1.0 / G0, "offset": 0.0,
        "refline": Z500_REF_GEOPOTENTIAL / G0, "detrend_rmst": True,
    },
    {
        "label": "May q min", "units": "g kg$^{-1}$", "hist_var": "q_may_min",
        "event_source": "q_min", "mode": "min", "scale": 1000.0, "offset": 0.0,
    },
]


def collect_violin_data():
    require_file(HIST_EXTREMES_FILE)
    require_file(COMPILED_FILE)

    data = {}
    with xr.open_dataset(HIST_EXTREMES_FILE) as hist_ds, xr.open_dataset(
        COMPILED_FILE
    ) as et_ds:
        et_ds = et_ds.sel(time=slice(EVENT_START_DATE, EVENT_END_DATE))

        for spec in VIOLIN_SPECS:
            hist_series = big_box_land_mean(hist_ds[spec["hist_var"]])

            source = et_ds[spec["event_source"]]
            if spec["mode"] == "max":
                event_extreme_field = source.max("time")
            else:
                event_extreme_field = source.min("time")

            event_raw = float(big_box_land_mean(event_extreme_field))
            hist_raw = np.asarray(hist_series.values, dtype=float).reshape(-1)

            # Analogue-consistent RMST detrend (z500 only): remove the warming
            # trend fit against the same RMST, referenced to the event-year
            # climate (event value unchanged; historical lifted to present).
            if DETREND_Z500_VIOLIN and spec.get("detrend_rmst"):
                rmst = analogue_rmst_cached()
                hist_years = _series_years(hist_series)
                rmst_year = np.array([_rmst_at_year(rmst, y) for y in hist_years])
                rmst_event = float(
                    rmst.sel(
                        time=np.datetime64(pd.Timestamp(TARGET_DAY).normalize()),
                        method="nearest",
                    )
                )
                ok = np.isfinite(hist_raw) & np.isfinite(rmst_year)
                slope = float(np.polyfit(rmst_year[ok], hist_raw[ok], 1)[0])
                hist_raw = hist_raw - slope * (rmst_year - rmst_event)
                print(f"[INFO] Z500 violin detrend: slope={slope:.1f} m2s-2/K "
                      f"({slope / G0:.2f} m/K); historical lifted to "
                      f"{pd.Timestamp(TARGET_DAY).year} climate (event unchanged)")

            scale = spec["scale"]
            offset = spec["offset"]
            data[spec["label"]] = {
                "historical": hist_raw * scale + offset,
                "et_extreme": event_raw * scale + offset,
            }
    return data


def plot_panel_violins(axes):
    data = collect_violin_data()
    rng = np.random.default_rng(42)

    for i, (ax, spec) in enumerate(zip(axes, VIOLIN_SPECS)):
        label = spec["label"]
        hist = np.asarray(data[label]["historical"], dtype=float)
        hist = hist[np.isfinite(hist)]

        parts = ax.violinplot([hist], positions=[0.0], widths=0.7, showextrema=False)
        for body in parts["bodies"]:
            body.set_facecolor("0.75")
            body.set_edgecolor("0.4")
            body.set_alpha(0.6)

        x_hist = rng.normal(loc=0.0, scale=0.035, size=len(hist))
        ax.scatter(x_hist, hist, s=18, color="0.35", alpha=0.6, edgecolor="none",
                   label=f"ERA5 May {START_YEAR}-{END_YEAR}")

        et_value = data[label]["et_extreme"]
        ax.scatter([0], [et_value], s=140, marker="*", color="#d62728",
                   edgecolor="black", linewidth=0.7, zorder=5)
        ax.annotate("2026", xy=(0, et_value), xytext=(0.14, et_value),
                    color="#d62728", fontsize=FS_TICK - 2, fontweight="bold",
                    va="center")

        if "refline" in spec:
            ax.axhline(spec["refline"], color="black", linestyle="--",
                       linewidth=1.0, zorder=1)

        ax.set_xlim(-0.45, 0.45)
        ax.set_xticks([])
        ax.set_ylabel(f"{label} [{spec['units']}]", fontsize=FS_LABEL)
        ax.tick_params(labelsize=FS_TICK)
        ax.grid(axis="y", color="0.88", linewidth=0.8)

        ax.text(0.5, 1.02, f"({chr(98 + i)})", transform=ax.transAxes,
                fontsize=FS_PANEL, fontweight="bold", va="bottom", ha="center")


def plot_panel_paris(ax):
    require_file(HOURLY_MSE_FILE)

    with xr.open_dataset(HOURLY_MSE_FILE) as ds:
        lat_name = coord_name(ds, ("lat", "latitude"))
        lon_name = coord_name(ds, ("lon", "longitude"))
        ds_paris_sel = ds.sel(
            {lat_name: PARIS_LAT, lon_name: PARIS_LON}, method="nearest"
        ).load()
        PARIS_LAT_SEL = ds_paris_sel[lat_name].values[()]
        PARIS_LON_SEL = ds_paris_sel[lon_name].values[()]
        ds_paris = ds.sel(
            {lat_name: slice(PARIS_LAT_SEL + 1, PARIS_LAT_SEL - 1),
             lon_name: slice(PARIS_LON_SEL - 1, PARIS_LON_SEL + 1)}
        ).load().mean([lat_name, lon_name])

    print(ds_paris)
    mse_kj = ds_paris["mse"]
    mse_sat_kj = ds_paris["mse_sat_500"]

    line_mse, = ax.plot(mse_kj["time"], mse_kj, color="#d62728", linewidth=1.3,
                        label="MSE (surface)")
    line_sat, = ax.plot(mse_sat_kj["time"], mse_sat_kj, color="#1f77b4",
                        linewidth=1.3, label="MSE$_{sat}$ (500 hPa)")
    handles = [line_mse, line_sat]

    day_start = pd.Timestamp(TARGET_DAY)
    ax.axvspan(day_start, day_start + pd.Timedelta(days=1), color="gold", alpha=0.25)

    t2m = None
    for candidate in ("t2m", "2t", "tas"):
        if candidate in ds_paris:
            t2m = ds_paris[candidate] - 273.15
            break

    if t2m is not None:
        ax2 = ax.twinx()
        line_t2m, = ax2.plot(t2m["time"], t2m, color="0.4", linewidth=1.0,
                             linestyle="--", label="T2m")
        ax2.set_ylabel("T2m [°C]", fontsize=FS_LABEL)
        ax2.tick_params(labelsize=FS_TICK)
        handles.append(line_t2m)
    else:
        print("[INFO] No hourly t2m in the hourly MSE file; add "
              "'t2m': t2m.astype('float32') to ds_out in the hourly MSE "
              "script and rerun to enable the second y-axis.")

    ax.set_xlim(pd.Timestamp(EVENT_START_DATE),
                pd.Timestamp(EVENT_END_DATE) + pd.Timedelta(days=1))
    ax.set_ylabel("Paris Moist Static Energy [J g$^{-1}$]", fontsize=FS_LABEL)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.tick_params(axis="x", rotation=30, labelsize=FS_TICK)
    ax.tick_params(axis="y", labelsize=FS_TICK)
    ax.grid(color="0.88", linewidth=0.8)
    ax.legend(handles=handles, loc="upper left", fontsize=FS_TICK, frameon=False)
    add_panel_label(ax, "(c)")


# =====================================================================
# Panels (c)+(d): IFS ensemble top-5 composites (Z500 & T2m), pooled
# across BOTH inits, shown as daily-mean T2m anomaly vs the ERA5
# climatology with the composite's absolute Z500 height contours.
# Self-contained: opens the IFS GRIBs and builds/loads the ERA5 t2m
# climatology itself (reuses CLIM_* / G0 / FS_* / helpers from above).
# =====================================================================
from matplotlib.lines import Line2D

COMP_T2M_GRIB_TMPL = (
    "${IFS_FORECAST_DIR}/"
    "tigge_ecmwf_pf_2m_temperature_sfc_6h_{date}_05x05.grib"   # 0.5x0.5 t2m, one file per init
)
COMP_Z500_GRIB_TMPL = (
    "${IFS_FORECAST_DIR}/"
    "tigge_ecmwf_pf_gh500_6h_{date}_1x1.grib"                  # 1x1 z500, one file per init
)

COMP_VALID_DATE = pd.Timestamp(TARGET_DAY)
COMP_INIT_DATES = [pd.Timestamp("2026-05-17"), pd.Timestamp("2026-05-18")]
COMP_N_TOP = 5
COMP_SAMPLE_HOURS = [0, 6, 12, 18]
COMP_MEMBER_DIM = "number"

# Target box (same western-Europe box as the rest of the figure)
COMP_BOX_LON_MIN, COMP_BOX_LON_MAX = LON_MIN, LON_MAX
COMP_BOX_LAT_MIN, COMP_BOX_LAT_MAX = LAT_MIN, LAT_MAX

# Broad plotting region for the composite maps
COMP_MAP_LON_MIN, COMP_MAP_LON_MAX = -20, 20
COMP_MAP_LAT_MIN, COMP_MAP_LAT_MAX = 30, 65

COMP_T_ANOM_LIMIT = 12   # colour range for the anomaly shading [degC]
COMP_T_ANOM_STEP = 1.0   # discrete colour step [degC]; 0 sits on a white bin

# ERA5 t2m daily-mean climatology (row-1 anomaly reference)
COMP_ERA5_HIST_T2M_GRIB = "/pool/data/ERA5/E5/sf/an/1H/167/E5sf00_1H_{date}_167.grb"
COMP_CLIM_LON_MIN, COMP_CLIM_LON_MAX = -32, 27
COMP_CLIM_LAT_MIN, COMP_CLIM_LAT_MAX = 23, 67
COMP_CLIM_TMP_DIR = (
    f"${SCRATCH}/era5_t2m_daymean_clim_{CLIM_START_YEAR}_{CLIM_END_YEAR}"
)
COMP_CLIM_CACHE_FILE = (
    f"${WORK}/02_MSE/western_europe/"
    f"era5_t2m_daymean_clim_{COMP_VALID_DATE:%m%d}_"
    f"{CLIM_START_YEAR}_{CLIM_END_YEAR}_w{2 * CLIM_HALF_WINDOW_DAYS + 1}d.nc"
)
COMP_N_JOBS_CLIM = 16


def comp_get_variable(ds, preferred_names):
    for name in preferred_names:
        if name in ds:
            return ds[name]
    return list(ds.data_vars.values())[0]


def comp_prepare_coordinates(data):
    lon_name = "longitude" if "longitude" in data.coords else "lon"
    lat_name = "latitude" if "latitude" in data.coords else "lat"
    data = data.assign_coords({lon_name: ((data[lon_name] + 180) % 360) - 180})
    data = data.sortby(lon_name).sortby(lat_name)
    return data, lon_name, lat_name


def comp_make_lat_slice(data, lat_name, south, north):
    if data[lat_name][0] > data[lat_name][-1]:
        return slice(north, south)
    return slice(south, north)


def comp_select_daily_field(data, initialization, valid):
    """Select one initialization and average its valid-day 6-hourly steps."""
    data_init = data.sel(time=initialization)
    if "valid_time" in data_init.coords:
        valid_times = data_init["valid_time"]
    else:
        valid_times = data_init["time"] + data_init["step"]
    valid_dates = pd.to_datetime(valid_times.values).floor("D")
    step_indices = np.where(valid_dates == valid.floor("D"))[0]
    if len(step_indices) == 0:
        raise ValueError(
            f"No field for init {initialization:%Y-%m-%d} valid {valid:%Y-%m-%d}"
        )
    daily = data_init.isel(step=step_indices)
    if len(step_indices) > 1:
        daily = daily.mean("step", skipna=True)
    else:
        daily = daily.isel(step=0)
    return daily


def comp_crop_map(data, lon_name, lat_name):
    return data.sel({
        lon_name: slice(COMP_MAP_LON_MIN, COMP_MAP_LON_MAX),
        lat_name: comp_make_lat_slice(
            data, lat_name, COMP_MAP_LAT_MIN, COMP_MAP_LAT_MAX
        ),
    })


def comp_target_box_mean(data, lon_name, lat_name):
    """cos(lat)-weighted box mean, each cell also weighted by the fraction of its
    area inside the target box -> grid-independent (no half-degree edge jumps)."""
    lon, lat = data[lon_name], data[lat_name]
    dlon = float(abs(lon.values[1] - lon.values[0]))
    dlat = float(abs(lat.values[1] - lat.values[0]))
    fx = ((np.minimum(lon + dlon/2, COMP_BOX_LON_MAX) -
           np.maximum(lon - dlon/2, COMP_BOX_LON_MIN)) / dlon).clip(0, 1)   # lon overlap fraction
    fy = ((np.minimum(lat + dlat/2, COMP_BOX_LAT_MAX) -
           np.maximum(lat - dlat/2, COMP_BOX_LAT_MIN)) / dlat).clip(0, 1)   # lat overlap fraction
    w = fx * fy * np.cos(np.deg2rad(lat))                                   # 2D via broadcasting
    valid = np.isfinite(data)
    return (data.where(valid) * w).sum((lon_name, lat_name)) / (w * valid).sum((lon_name, lat_name))


def comp_to_celsius(field):
    units = field.attrs.get("units", "").lower()
    if units in ["k", "kelvin"] or float(field.mean()) > 100:
        field = field - 273.15
    field.attrs["units"] = "°C"
    return field


def comp_prepare_z500_day(field, z_lon, z_lat, convert_to_height):
    field = comp_crop_map(field, z_lon, z_lat)
    if convert_to_height:
        field = field / G0
    field.attrs["units"] = "m"
    return field


def comp_regrid_era5_to_ifs(field, ref, ref_lon, ref_lat):
    era_lon = "longitude" if "longitude" in field.coords else "lon"
    era_lat = "latitude" if "latitude" in field.coords else "lat"
    field = (
        field.assign_coords({era_lon: ((field[era_lon] + 180) % 360) - 180})
        .sortby(era_lon)
        .sortby(era_lat)
    )
    rename_coords = {}
    if era_lon != ref_lon:
        rename_coords[era_lon] = ref_lon
    if era_lat != ref_lat:
        rename_coords[era_lat] = ref_lat
    if rename_coords:
        field = field.rename(rename_coords)
    return field.interp(
        {ref_lon: ref[ref_lon], ref_lat: ref[ref_lat]}, method="linear"
    )


def comp_climatology_window_days():
    days = []
    for year in range(CLIM_START_YEAR, CLIM_END_YEAR + 1):
        center = COMP_VALID_DATE.replace(year=year)
        for offset in range(-CLIM_HALF_WINDOW_DAYS, CLIM_HALF_WINDOW_DAYS + 1):
            days.append(center + pd.Timedelta(days=offset))
    return days


def comp_process_climatology_day(day):
    """One CDO call: hourly E5 GRIB -> 4-point daily-mean t2m region box."""
    from cdo import Cdo

    out_nc = os.path.join(COMP_CLIM_TMP_DIR, f"t2m_daymean_{day:%Y-%m-%d}.nc")
    if os.path.exists(out_nc):
        return out_nc
    grib_file = COMP_ERA5_HIST_T2M_GRIB.format(date=f"{day:%Y-%m-%d}")
    if not os.path.exists(grib_file):
        print(f"[WARN] Missing ERA5 GRIB, skipping: {grib_file}")
        return None
    hours = ",".join(str(h) for h in COMP_SAMPLE_HOURS)
    Cdo().daymean(
        input=(
            f"-sellonlatbox,{COMP_CLIM_LON_MIN},{COMP_CLIM_LON_MAX},"
            f"{COMP_CLIM_LAT_MIN},{COMP_CLIM_LAT_MAX} "
            f"-setgridtype,regular -selhour,{hours} {grib_file}"
        ),
        output=out_nc,
        options="-O -f nc --eccodes",
    )
    return out_nc


def comp_build_t2m_daymean_climatology():
    """ERA5 t2m daily-mean climatology for TARGET_DAY's calendar day, cached."""
    if os.path.exists(COMP_CLIM_CACHE_FILE):
        print(f"[INFO] Loading cached t2m climatology: {COMP_CLIM_CACHE_FILE}")
        with xr.open_dataset(COMP_CLIM_CACHE_FILE) as ds:
            return ds["t2m_clim"].load()

    # Build it (Levante CDO + parallel workers); only needed if no cache.
    from joblib import Parallel, delayed

    os.environ["PATH"] = (
        "/path/to/cdo/bin"
        + os.pathsep
        + os.environ.get("PATH", "")
    )
    os.makedirs(COMP_CLIM_TMP_DIR, exist_ok=True)
    days = comp_climatology_window_days()
    print(
        f"[INFO] Building t2m climatology from {len(days)} days "
        f"({CLIM_START_YEAR}-{CLIM_END_YEAR}, "
        f"+-{CLIM_HALF_WINDOW_DAYS} d around {COMP_VALID_DATE:%d %b})"
    )
    day_files = Parallel(n_jobs=COMP_N_JOBS_CLIM, verbose=5)(
        delayed(comp_process_climatology_day)(day) for day in days
    )
    valid_files = [p for p in day_files if p is not None]
    if not valid_files:
        raise RuntimeError("No ERA5 climatology days could be processed.")
    with xr.open_mfdataset(
        sorted(valid_files), combine="nested", concat_dim="time"
    ) as ds:
        t2m_days = comp_get_variable(ds, ["2t", "t2m", "var167"])
        clim = t2m_days.mean("time", skipna=True).load()
    clim.name = "t2m_clim"
    clim.attrs.update({"long_name": "ERA5 daily-mean t2m climatology", "units": "K"})
    clim.to_dataset().to_netcdf(COMP_CLIM_CACHE_FILE)
    print(f"[DONE] Cached t2m climatology: {COMP_CLIM_CACHE_FILE}")
    return clim


def comp_open_inits(tmpl, inits, var_names):
    """Open one per-day TIGGE file per init and stack them along a 'time' (init) dim,
    reproducing the old monthly-file layout so downstream selection (data.sel(time=init))
    is unchanged. Handles the new per-day files (t2m 0.5x0.5, z500 1x1)."""
    das = []
    for init in inits:
        path = tmpl.format(date=f"{init:%Y%m%d}")
        require_file(path)
        ds = xr.open_dataset(path, engine="cfgrib",
                             backend_kwargs={"indexpath": path + ".idx"})
        da = comp_get_variable(ds, var_names)
        if "time" not in da.dims:
            da = da.expand_dims("time")
        da = da.drop_vars("valid_time", errors="ignore")  # recompute time+step per init (no concat ambiguity)
        das.append(da)
    return comp_prepare_coordinates(xr.concat(das, dim="time"))


def plot_panel_composite(fig, ax_z500, ax_t2m):
    # --- Open IFS ensemble t2m & z500 ---
    # per-day files: t2m 0.5x0.5, z500 1x1, one per init, stacked along 'time'
    t2m, t_lon, t_lat = comp_open_inits(COMP_T2M_GRIB_TMPL, COMP_INIT_DATES, ["t2m", "2t"])
    z500, z_lon, z_lat = comp_open_inits(COMP_Z500_GRIB_TMPL, COMP_INIT_DATES, ["gh", "z"])
    z500_source_name = z500.name
    z_units = z500.attrs.get("units", "").lower().replace(" ", "")
    convert_z500_to_height = (
        "m2s-2" in z_units or "m**2s**-2" in z_units
        or "m^2s^-2" in z_units or z500_source_name == "z"
    )

    # --- Per-init daily-mean fields (cropped), kept for compositing ---
    t2m_day_by_init, z500_day_by_init = {}, {}
    for panel_init in COMP_INIT_DATES:
        key = f"{panel_init:%Y-%m-%d}"
        t2m_day_by_init[key] = comp_to_celsius(
            comp_crop_map(
                comp_select_daily_field(t2m, panel_init, COMP_VALID_DATE),
                t_lon, t_lat,
            )
        )
        z500_day_by_init[key] = comp_prepare_z500_day(
            comp_select_daily_field(z500, panel_init, COMP_VALID_DATE),
            z_lon, z_lat, convert_z500_to_height,
        )

    # --- Pool every (init, member) and rank by target-box daily mean ---
    frames = []
    for panel_init in COMP_INIT_DATES:
        key = f"{panel_init:%Y-%m-%d}"
        t2m_box = comp_target_box_mean(t2m_day_by_init[key], t_lon, t_lat).to_series()
        z500_box = comp_target_box_mean(z500_day_by_init[key], z_lon, z_lat).to_series()
        part = pd.DataFrame({"t2m_C": t2m_box, "z500_m": z500_box})
        part.index.name = "member"
        part = part.reset_index()
        part.insert(0, "init", key)
        frames.append(part)
    pooled = pd.concat(frames, ignore_index=True)
    top_t2m = pooled.sort_values("t2m_C", ascending=False).head(COMP_N_TOP)
    _hot_ids = set(map(tuple, top_t2m[["init", "member"]].to_numpy()))          # members already in top-T2m
    top_z500 = (pooled[~pooled[["init", "member"]].apply(tuple, axis=1).isin(_hot_ids)]
                .sort_values("z500_m", ascending=False).head(COMP_N_TOP))         # exclusive by construction

    # --- ALL composite-pool members (both inits): target-box daily-mean T2m & Z500, with top-5 flags ---
    _ztop_ids = set(map(tuple, top_z500[["init", "member"]].to_numpy()))
    _allm = pooled.copy()
    _tup = _allm[["init", "member"]].apply(tuple, axis=1)
    _allm["in_top5_t2m"]  = _tup.isin(_hot_ids)
    _allm["in_top5_z500"] = _tup.isin(_ztop_ids)
    _allm = _allm.sort_values(["init", "member"]).reset_index(drop=True)
    _allm_csv = f"composite_all_members_t2m_z500_{COMP_VALID_DATE:%Y%m%d}.csv"
    _allm.to_csv(_allm_csv, index=False)
    print(f"[composite] wrote {_allm_csv}: {len(_allm)} members "
          f"({int(_allm['in_top5_t2m'].sum())} top-T2m, {int(_allm['in_top5_z500'].sum())} top-Z500)")
    print(_allm.to_string(index=False))

    print(f"Top-{COMP_N_TOP} T2m composite members (both inits pooled):")
    for rank, r in enumerate(top_t2m.itertuples(index=False), 1):
        print(f"  {rank}. init {r.init}  member {int(r.member):>2}  {r.t2m_C:6.2f} °C")
    print(f"Top-{COMP_N_TOP} Z500 composite members (excl. top-T2m, both inits pooled):")
    for rank, r in enumerate(top_z500.itertuples(index=False), 1):
        print(f"  {rank}. init {r.init}  member {int(r.member):>2}  {r.z500_m:7.1f} m")

    def composite_fields(selected, field_by_init):
        picked = [
            field_by_init[r.init].sel({COMP_MEMBER_DIM: int(r.member)})
            for r in selected.itertuples(index=False)
        ]
        return xr.concat(picked, dim="picked").mean("picked", skipna=True)

    t_high_t2m = composite_fields(top_t2m, t2m_day_by_init)
    z_high_t2m = composite_fields(top_t2m, z500_day_by_init)
    t_high_z500 = composite_fields(top_z500, t2m_day_by_init)
    z_high_z500 = composite_fields(top_z500, z500_day_by_init)

    # --- ERA5 t2m climatology (anomaly reference), remapped to the IFS grid ---
    # Anomaly reference: the SAME ERA5 0.5-deg daily-mean t2m product as panel (a),
    # regridded onto the IFS 0.5x0.5 t2m grid so the two are definitely matched.
    era5_t2m_clim = era5_t2m_0p5_climatology()                    # already degC, 0.5deg ERA5 grid
    era5_t2m_clim_ifs = comp_regrid_era5_to_ifs(
        era5_t2m_clim, t_high_z500, t_lon, t_lat
    )

    # --- Shared absolute z500 contour levels across both composites ---
    z_fields = [z_high_z500, z_high_t2m]
    z_min = min(float(f.min(skipna=True)) for f in z_fields)
    z_max = max(float(f.max(skipna=True)) for f in z_fields)
    z500_levels = np.arange(
        np.floor(z_min / 40) * 40, np.ceil(z_max / 40) * 40 + 40, 40
    )

    # Discrete anomaly scale: 1-degC steps, symmetric about 0, 0 on a white bin.
    anom_norm, cmap, anom_levels = anomaly_discrete_scale(
        COMP_T_ANOM_LIMIT, COMP_T_ANOM_STEP
    )

    panels = [
        (ax_z500, t_high_t2m,  z_high_t2m,  f"IFS {COMP_N_TOP} hottest", "(e)"),                    # left
        (ax_t2m,  t_high_z500, z_high_z500, f"IFS {COMP_N_TOP} highest-Z500", "(f)"),  # right
    ]
    mesh = None
    for ax, t_field, z_field, legend_title, label in panels:
        t_anom = t_field - era5_t2m_clim_ifs+273.15
        mesh = ax.pcolormesh(
            t_anom[t_lon], t_anom[t_lat], t_anom, cmap=cmap, norm=anom_norm,
            shading="auto", transform=ccrs.PlateCarree(),
        )
        cs = ax.contour(
            z_field[z_lon], z_field[z_lat], z_field, levels=z500_levels,
            colors="black", linewidths=0.9, transform=ccrs.PlateCarree(),
        )
        ax.clabel(cs, inline=True, fontsize=FS_CLABEL, fmt="%d")

        ax.set_extent(
            [COMP_MAP_LON_MIN, COMP_MAP_LON_MAX, COMP_MAP_LAT_MIN, COMP_MAP_LAT_MAX],
            crs=ccrs.PlateCarree(),
        )
        ax.coastlines(resolution="10m", linewidth=0.6)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4)
        ax.plot(
            [COMP_BOX_LON_MIN, COMP_BOX_LON_MAX, COMP_BOX_LON_MAX,
             COMP_BOX_LON_MIN, COMP_BOX_LON_MIN],
            [COMP_BOX_LAT_MIN, COMP_BOX_LAT_MIN, COMP_BOX_LAT_MAX,
             COMP_BOX_LAT_MAX, COMP_BOX_LAT_MIN],
            color="black", linestyle="--", linewidth=1.8,
            transform=ccrs.PlateCarree(), zorder=6,
        )
        gl = ax.gridlines(draw_labels=True, linestyle="--", alpha=0.5)
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {"size": FS_TICK}
        gl.ylabel_style = {"size": FS_TICK}

        target_anom = float(comp_target_box_mean(t_anom, t_lon, t_lat))
        target_z = float(comp_target_box_mean(z_field, z_lon, z_lat))
        ax.legend(
            handles=[
                Line2D([0], [0], color="none",
                       label=f"box T2m anom: {target_anom:+.2f} °C"),
                Line2D([0], [0], color="none",
                       label=f"box Z500: {target_z:.1f} m"),
            ],
            loc="lower right", title=legend_title, framealpha=0.9,
            fontsize=FS_TICK - 1, title_fontsize=FS_TICK - 1,
            handlelength=0, handletextpad=0,
        )
        add_panel_label(ax, label)

        cb = fig.colorbar(
            mesh, ax=ax, orientation="vertical", shrink=0.85, pad=0.02,
            ticks=anom_levels[::2],
        )
        cb.set_label(
            f"IFS Daily Mean 2m Temp. Anomaly [°C]",
            fontsize=FS_LABEL,
        )
        cb.ax.tick_params(labelsize=FS_TICK)


def main():
    fig = plt.figure(figsize=(15, 11))
    gs = fig.add_gridspec(2, 2, hspace=0.28, wspace=0.18)

    ax1 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
    plot_panel_map(fig, ax1)

    gs2 = gs[0, 1].subgridspec(1, 3, wspace=0.75)
    axes2 = [fig.add_subplot(gs2[0, i]) for i in range(3)]
    plot_panel_violins(axes2)

    ax3 = fig.add_subplot(gs[1, 0], projection=ccrs.PlateCarree())
    ax4 = fig.add_subplot(gs[1, 1], projection=ccrs.PlateCarree())
    plot_panel_composite(fig, ax3, ax4)

    fig.savefig(OUT_PNG, dpi=220, bbox_inches="tight")
    print(f"[DONE] Wrote figure: {OUT_PNG}")
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
# ===== CSV: daily-mean box T2m & Z500 for all members (both inits) =====
import pandas as pd

t2m,  t_lon, t_lat = comp_open_inits(COMP_T2M_GRIB_TMPL,  COMP_INIT_DATES, ["t2m", "2t"])   # 0.5°
z500, z_lon, z_lat = comp_open_inits(COMP_Z500_GRIB_TMPL, COMP_INIT_DATES, ["gh", "z"])     # 1°
zu = z500.attrs.get("units", "").lower().replace(" ", "")
zconv = ("m2s-2" in zu or "m**2s**-2" in zu or "m^2s^-2" in zu or z500.name == "z")

rows = []
for init in COMP_INIT_DATES:
    t_day = comp_to_celsius(comp_crop_map(comp_select_daily_field(t2m, init, COMP_VALID_DATE), t_lon, t_lat))
    z_day = comp_prepare_z500_day(comp_select_daily_field(z500, init, COMP_VALID_DATE), z_lon, z_lat, zconv)
    part = pd.DataFrame({
        "t2m_C":  comp_target_box_mean(t_day, t_lon, t_lat).to_series(),
        "z500_m": comp_target_box_mean(z_day, z_lon, z_lat).to_series(),
    })
    part.index.name = "member"
    part = part.reset_index()
    part.insert(0, "init", f"{init:%Y-%m-%d}")
    rows.append(part)

out = pd.concat(rows, ignore_index=True)
out.insert(1, "valid_date", COMP_VALID_DATE.strftime("%Y-%m-%d"))
out = out[["init", "valid_date", "member", "t2m_C", "z500_m"]].sort_values(["init", "member"])

csv_path = f"ifs_box_daily_t2m_z500_{COMP_VALID_DATE:%Y%m%d}.csv"
out.to_csv(csv_path, index=False)
print(f"wrote {csv_path}  ({len(out)} rows, {out['member'].nunique()} members × {len(COMP_INIT_DATES)} inits)")
print(out.head(8).to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import nicopal as ncp
import glob, os

sequential_cmap = ncp.pal("Boron")
diverging_cmap = ncp.pal("Cobalt")

# =========================
# Combined figure (all rows based on DAILY-MEAN z500 from 6-hourly steps).
# Box-mean (rows 1-3) uses the grid-independent fractional cos-lat box mean
# (comp_target_box_mean, identical to the Figure 1 script) instead of a whole-cell mean.
# =========================

g0 = 9.80665

init_dim = "time"
lead_dim = "step"
member_dim = "number"
daily_dim = "lead_day"

steps_per_day = 4

# Target ("big") box used for the area-mean plume and the probability rows
box_lon_min, box_lon_max = -5.5, 5
box_lat_min, box_lat_max = 44.5, 55

# Broad plotting region for the bottom-row map extent
map_lon_min, map_lon_max = -30, 25
map_lat_min, map_lat_max = 25, 65

threshold = 5860.0  # m (geopotential height)

init_start = pd.Timestamp("2026-05-15")
plot_start = pd.Timestamp("2026-05-15")
plot_end = pd.Timestamp("2026-06-05")

highlight_init_date = pd.Timestamp("2026-05-18")

target_valid_date = pd.Timestamp("2026-05-25")
init_dates_to_plot = [
    pd.Timestamp("2026-05-17"),
    pd.Timestamp("2026-05-18"),
    pd.Timestamp("2026-05-19"),
]
max_init_b = pd.Timestamp("2026-05-25")

# =========================
# Target-region mean: copied verbatim from comp_target_box_mean in the Figure 1
# script (same box constants), so box values match Fig. 1 exactly.
# =========================
COMP_BOX_LON_MIN, COMP_BOX_LON_MAX = box_lon_min, box_lon_max
COMP_BOX_LAT_MIN, COMP_BOX_LAT_MAX = box_lat_min, box_lat_max


def comp_target_box_mean(data, lon_name, lat_name):
    """cos(lat)-weighted box mean, each cell also weighted by the fraction of its
    area inside the target box -> grid-independent (no half-degree edge jumps)."""
    lon, lat = data[lon_name], data[lat_name]
    dlon = float(abs(lon.values[1] - lon.values[0]))
    dlat = float(abs(lat.values[1] - lat.values[0]))
    fx = ((np.minimum(lon + dlon/2, COMP_BOX_LON_MAX) -
           np.maximum(lon - dlon/2, COMP_BOX_LON_MIN)) / dlon).clip(0, 1)   # lon overlap fraction
    fy = ((np.minimum(lat + dlat/2, COMP_BOX_LAT_MAX) -
           np.maximum(lat - dlat/2, COMP_BOX_LAT_MIN)) / dlat).clip(0, 1)   # lat overlap fraction
    w = fx * fy * np.cos(np.deg2rad(lat))                                   # 2D via broadcasting
    valid = np.isfinite(data)
    return (data.where(valid) * w).sum((lon_name, lat_name)) / (w * valid).sum((lon_name, lat_name))


# =========================
# Open z500-only GRIB and prepare the field
# =========================

# --- z500 1x1 files: ONE GRIB per init date (same files as the Figure 1 composites) ---
Z500_DIR = "${IFS_FORECAST_DIR}/"
Z500_GLOB = os.path.join(Z500_DIR, "tigge_ecmwf_pf_gh500_6h_*_1x1.grib")

# Memory: each file is a global 50-member field for every lead time. Concatenating
# them uncropped loads all of it into RAM (this is what crashed the kernel), so each
# file is cropped to the region / inits / lead times the figure actually uses and
# loaded on its own before concatenation.
pad = 2.0   # keeps the boundary-straddling cells of the target box (partial weights)
crop_lon = (min(map_lon_min, box_lon_min - pad), max(map_lon_max, box_lon_max + pad))
crop_lat = (min(map_lat_min, box_lat_min - pad), max(map_lat_max, box_lat_max + pad))
last_init = max(max_init_b, target_valid_date)
last_valid = plot_end + pd.Timedelta(days=1)     # keep steps valid before this time


def _get_z(dsi):
    if "z" in dsi:  return dsi["z"]
    if "gh" in dsi: return dsi["gh"]
    return list(dsi.data_vars.values())[0]

files = sorted(glob.glob(Z500_GLOB))
if not files:
    raise FileNotFoundError(f"No z500 1x1 files matching {Z500_GLOB}")

das = []
for f in files:
    dsi = xr.open_dataset(f, engine="cfgrib", backend_kwargs={"indexpath": f + ".idx"})
    zi = _get_z(dsi)
    if "time" not in zi.dims:                          # per-file 'time' = the (scalar) init date
        zi = zi.expand_dims("time")
    init = pd.Timestamp(zi["time"].values[0])
    if init < init_start or init > last_init:
        dsi.close()
        continue
    zi = zi.drop_vars("valid_time", errors="ignore")   # recomputed downstream as time + step
    f_lon = "longitude" if "longitude" in zi.coords else "lon"
    f_lat = "latitude" if "latitude" in zi.coords else "lat"
    zi = zi.assign_coords({f_lon: ((zi[f_lon] + 180) % 360) - 180}).sortby(f_lon).sortby(f_lat)
    zi = zi.sel({f_lon: slice(*crop_lon), f_lat: slice(*crop_lat)})
    zi = zi.isel({lead_dim: np.where((init + pd.to_timedelta(zi[lead_dim].values)) < last_valid)[0]})
    das.append(zi.load().astype("float32"))
    dsi.close()
if not das:
    raise ValueError(f"No z500 files with inits between {init_start.date()} and {last_init.date()}")
print(f"[INFO] Loaded {len(das)} per-init z500 1x1 files (cropped)")
z = xr.concat(das, dim="time").sortby("time")

# Convert geopotential -> geopotential height (m) if needed
z_source_name = z.name
units = z.attrs.get("units", "").lower().replace(" ", "")
convert_to_height = (
    "m2s-2" in units or "m**2s**-2" in units
    or "m^2s^-2" in units or z_source_name == "z"
)
if convert_to_height:
    z = z / g0
z.attrs["units"] = "m"

lon_name = "longitude" if "longitude" in z.coords else "lon"
lat_name = "latitude" if "latitude" in z.coords else "lat"
z = z.assign_coords({lon_name: ((z[lon_name] + 180) % 360) - 180})
z = z.sortby(lon_name).sortby(lat_name)


# =========================
# Daily-mean selection of the raw field for one init and one valid day
# =========================

def select_daily_steps(z_field, init_date, valid_date):
    z_init = z_field.sel({init_dim: init_date})
    if "valid_time" in z_init.coords:
        vt = z_init["valid_time"]
    else:
        vt = z_init[init_dim] + z_init[lead_dim]
    vt_dates = pd.to_datetime(vt.values).floor("D")
    step_index = np.where(vt_dates == valid_date.floor("D"))[0]
    if len(step_index) == 0:
        raise ValueError(
            f"No forecast valid on {valid_date.date()} for init {init_date.date()}"
        )
    z_valid = z_init.isel({lead_dim: step_index})
    if lead_dim in z_valid.dims:
        if z_valid.sizes[lead_dim] > 1:
            z_valid = z_valid.mean(lead_dim, skipna=True)
        else:
            z_valid = z_valid.isel({lead_dim: 0})
    return z_valid


def get_ensmean_for_init_and_valid(z_field, init_date, valid_date):
    return select_daily_steps(z_field, init_date, valid_date).mean(member_dim, skipna=True)


# =========================
# Big-box regional mean (rows 1-3) — fractional cos(lat) box mean.
# A small pad keeps the boundary-straddling cells so their partial weights count.
# =========================

z_box = z.sel(
    {
        lon_name: slice(box_lon_min - pad, box_lon_max + pad),
        lat_name: slice(box_lat_min - pad, box_lat_max + pad),
    }
)
z500_mean_6h = comp_target_box_mean(z_box, lon_name, lat_name)
z500_mean_6h.name = "z500_big_box_mean"

init_dates_all = pd.to_datetime(z500_mean_6h[init_dim].values)
keep_inits = init_dates_all >= init_start
z500_mean_6h = z500_mean_6h.isel({init_dim: keep_inits})
init_dates = pd.to_datetime(z500_mean_6h[init_dim].values)
plume_inits = init_dates[init_dates <= max_init_b]   # panel (a): same inits as panel (b)


# =========================
# Collapse the 6-hourly steps to daily means
# =========================

step_hours = (z500_mean_6h[lead_dim] / np.timedelta64(1, "h")).astype(int).values
lead_day_values = step_hours // 24
z500_mean_6h = z500_mean_6h.assign_coords({daily_dim: (lead_dim, lead_day_values)})

unique_days, day_counts = np.unique(lead_day_values, return_counts=True)
complete_days = unique_days[day_counts == steps_per_day]
dropped_days = unique_days[day_counts != steps_per_day]
if len(dropped_days) > 0:
    print(f"[INFO] Dropping incomplete valid day(s) at lead(s) "
          f"{list(dropped_days)} (fewer than {steps_per_day} steps)")

z500_mean = z500_mean_6h.groupby(daily_dim).mean(dim=lead_dim)
z500_mean = z500_mean.sel({daily_dim: complete_days})
print(f"[INFO] Daily means built from {steps_per_day} steps/day; "
      f"{z500_mean.sizes[daily_dim]} complete valid days per initialization")

valid_time = (
    z500_mean[init_dim] + z500_mean[daily_dim] * np.timedelta64(1, "D")
).transpose(init_dim, daily_dim)


# =========================
# Change in exceedance probability (row 2)
# =========================

z500_prob = (z500_mean > threshold).mean(dim=member_dim, skipna=True) * 100
z500_prob = z500_prob.transpose(init_dim, daily_dim)

prob_init_dates = pd.to_datetime(z500_prob[init_dim].values)
lead_delta = pd.to_timedelta(z500_prob[daily_dim].values, unit="D")
prob_valid_time_grid = prob_init_dates.values[:, None] + lead_delta.values[None, :]

prob_df = pd.DataFrame(
    {
        "init_time": np.repeat(prob_init_dates.values, z500_prob.sizes[daily_dim]),
        "valid_time": prob_valid_time_grid.ravel(),
        "prob": z500_prob.values.ravel(),
    }
)
prob_wide = prob_df.pivot_table(index="init_time", columns="valid_time", values="prob")
dprob_wide = prob_wide.diff(axis=0).iloc[1:]
dprob_wide = dprob_wide.loc[dprob_wide.index <= max_init_b]   # <-- limit panel (b) to inits up to 23 May

dprob_x_dates = pd.to_datetime(dprob_wide.columns)
dprob_y_dates = pd.to_datetime(dprob_wide.index)
dprob_x = np.repeat(mdates.date2num(dprob_x_dates.to_pydatetime())[None, :], len(dprob_y_dates), axis=0)
dprob_y = np.repeat(mdates.date2num(dprob_y_dates.to_pydatetime())[:, None], len(dprob_x_dates), axis=1)

dprob_abs_max = float(np.nanmax(np.abs(dprob_wide.values)))
if not np.isfinite(dprob_abs_max) or dprob_abs_max == 0:
    dprob_abs_max = 1.0
dprob_lim = 10.0 * np.ceil(dprob_abs_max / 10.0)   # round colour limit to a multiple of 10 pp
dprob_norm = mcolors.TwoSlopeNorm(vmin=-dprob_lim, vcenter=0, vmax=dprob_lim)

print("Probability-change color scale: +/-", f"{dprob_abs_max:.1f}", "percentage points")
print("Maximum probability increase:", np.nanmax(dprob_wide.values), "percentage points")
print("Maximum probability decrease:", np.nanmin(dprob_wide.values), "percentage points")


# =========================
# Across-member std of the daily-mean box z500 (row 3)
# =========================

z500_std = z500_mean.std(dim=member_dim, skipna=True)
z500_frac = (z500_mean > threshold).mean(dim=member_dim, skipna=True)
std_init_dates = pd.to_datetime(z500_std[init_dim].values)

std_x_inits = []; std_values = []; frac_values = []
for init_date in std_init_dates:
    vt = pd.to_datetime(valid_time.sel({init_dim: init_date}).values).floor("D")
    day_index = np.where(vt == target_valid_date.floor("D"))[0]
    if len(day_index) == 0:
        continue
    std_values.append(float(z500_std.sel({init_dim: init_date}).isel({daily_dim: day_index[0]}).values))
    frac_values.append(float(z500_frac.sel({init_dim: init_date}).isel({daily_dim: day_index[0]}).values))
    std_x_inits.append(init_date)

std_x_inits = pd.to_datetime(std_x_inits)
std_values = np.array(std_values); frac_values = np.array(frac_values)
print("Across-member std of daily-mean box z500 valid", target_valid_date.date(),
      "ranges from", f"{np.nanmin(std_values):.1f}", "to", f"{np.nanmax(std_values):.1f}",
      "m over", len(std_values), "initializations")


# =========================
# Ensemble-mean daily-mean maps (row 4)
# =========================

maps = [get_ensmean_for_init_and_valid(z, init_date, target_valid_date) for init_date in init_dates_to_plot]
all_map_values = xr.concat(maps, dim="case")
map_region = all_map_values.sel(
    {lon_name: slice(map_lon_min, map_lon_max), lat_name: slice(map_lat_min, map_lat_max)}
)
map_vmin = float(map_region.min()); map_vmax = float(map_region.max())
# Round levels: 20 m shading, 40 m contours (labels are round numbers)
map_fill_dz, map_line_dz = 20.0, 40.0
map_lo = map_line_dz * np.floor(map_vmin / map_line_dz)
map_hi = map_line_dz * np.ceil(map_vmax / map_line_dz)
map_fill_levels = np.arange(map_lo, map_hi + map_fill_dz / 2, map_fill_dz)
map_line_levels = np.arange(map_lo, map_hi + map_line_dz / 2, map_line_dz)


# =========================
# Quick diagnostics (map-region peaks)
# =========================

def clip_to_map(field):
    return field.sel({lon_name: slice(map_lon_min, map_lon_max), lat_name: slice(map_lat_min, map_lat_max)})

raw_max = -np.inf; raw_max_init = None
for init_date in init_dates:
    try:
        field = select_daily_steps(z, init_date, target_valid_date)
    except ValueError:
        continue
    field_max = float(clip_to_map(field).max())
    if field_max > raw_max:
        raw_max = field_max; raw_max_init = init_date
if np.isfinite(raw_max):
    print(f"Max daily-mean z500 valid {target_valid_date.date()} "
          f"(any member, any init, map region): {raw_max:.1f} m (peak from init {raw_max_init.date()})")
else:
    print(f"No forecast valid on {target_valid_date.date()} among the available initializations.")

try:
    ensmean_init_field = get_ensmean_for_init_and_valid(z, target_valid_date, target_valid_date)
    ensmean_init_max = float(clip_to_map(ensmean_init_field).max())
    print(f"Max daily-mean z500 of ensemble mean initialized "
          f"{target_valid_date.date()} (map region): {ensmean_init_max:.1f} m")
except (ValueError, KeyError):
    print(f"No initialization on {target_valid_date.date()} in the dataset, "
          "so the ensemble-mean-initialized-that-day max is unavailable.")


# =========================
# ERA5(T) observed daily-mean box z500 (from 6-hourly steps) for panel (a)
# Same fractional cos-lat box mean and same 00/06/12/18 -> daily collapse
# used for the ensemble, so the black dashed obs line is directly comparable.
# =========================

era5_hourly_mse_file = (
    "${WORK}/02_MSE/western_europe/"
    "era5t_events/era5t_2026-05-15_to_2026-06-01/"
    "era5t_western_europe_2026-05-15_to_2026-06-01_hourly_mse.nc"
)

era5_ds = xr.open_dataset(era5_hourly_mse_file)
for _cand in ("z500", "gh500", "gh", "z"):
    if _cand in era5_ds:
        era5_da = era5_ds[_cand]
        break
else:
    era5_da = list(era5_ds.data_vars.values())[0]

# If a vertical dimension is present, pick the 500 hPa level
for _lev in ("level", "isobaricInhPa", "plev", "pressure_level"):
    if _lev in era5_da.dims:
        _lv = era5_da[_lev]
        _target = 50000.0 if float(_lv.max()) > 2000 else 500.0  # Pa vs hPa
        era5_da = era5_da.sel({_lev: _target}, method="nearest")
        break

# Geopotential -> geopotential height (m) if needed
_eunits = era5_da.attrs.get("units", "").lower().replace(" ", "")
if ("m2s-2" in _eunits or "m**2s**-2" in _eunits or "m^2s^-2" in _eunits
        or era5_da.name == "z"):
    era5_da = era5_da / g0

elon_name = "longitude" if "longitude" in era5_da.coords else "lon"
elat_name = "latitude" if "latitude" in era5_da.coords else "lat"
etime_name = "time" if "time" in era5_da.dims else (
    "valid_time" if "valid_time" in era5_da.dims else list(era5_da.dims)[0])

era5_da = era5_da.assign_coords({elon_name: ((era5_da[elon_name] + 180) % 360) - 180})
era5_da = era5_da.sortby(elon_name).sortby(elat_name)

era5_box = era5_da.sel(
    {
        elon_name: slice(box_lon_min - pad, box_lon_max + pad),
        elat_name: slice(box_lat_min - pad, box_lat_max + pad),
    }
)
era5_boxmean = comp_target_box_mean(era5_box.load(), elon_name, elat_name)

# Keep only 00/06/12/18 UTC, then daily-mean over days with all 4 steps
era5_times = pd.to_datetime(era5_boxmean[etime_name].values)
six_hourly = np.isin(era5_times.hour, [0, 6, 12, 18])
era5_boxmean = era5_boxmean.isel({etime_name: six_hourly})

era5_series = era5_boxmean.to_series()
era5_series.index = pd.to_datetime(era5_series.index).floor("D")
era5_day_counts = era5_series.groupby(level=0).count()
era5_daily = era5_series.groupby(level=0).mean()
era5_daily = era5_daily[era5_day_counts == steps_per_day]
print(f"[INFO] ERA5(T) daily-mean box z500 built from {steps_per_day} steps/day; "
      f"{len(era5_daily)} complete days")

# =========================
# Combined figure
# Sized for a full-width (17 cm) figure: 9 in wide, so fonts print at ~7-8 pt.
# Layout: (a) plume and (b) probability change share the valid-date axis;
# (c) spread/exceedance for the target day in its own row; (d)-(f) maps below.
# =========================

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.linewidth": 0.7,
})


def add_panel_label(ax, label, inset=False):
    """Bold panel letter outside the axes, above the top-left corner
    (inset=True: flush with the left edge, for tightly packed map panels)."""
    ax.annotate(f"({label})", xy=(0, 0.97), xycoords="axes fraction",
                xytext=(0, 5) if inset else (-4, 5), textcoords="offset points",
                ha="left" if inset else "right", va="bottom", fontsize=12, fontweight="bold",
                annotation_clip=False)


def set_date_axis(ax, start, end, label_every=2, minor=True):
    """Labelled ticks every `label_every` days ('15 May'), unlabelled daily minor ticks."""
    major = pd.date_range(start, end, freq=f"{label_every}D")
    ax.set_xticks(major)
    ax.set_xticklabels(major.strftime("%-d %b"))
    if minor:
        ax.set_xticks(pd.date_range(start, end, freq="1D"), minor=True)


fig = plt.figure(figsize=(9, 8.2))
outer = fig.add_gridspec(3, 1, height_ratios=[2.0, 0.62, 1.05], hspace=0.30,
                         left=0.08, right=0.92, top=0.97, bottom=0.05)
gs_top = outer[0].subgridspec(2, 2, width_ratios=[1, 0.022], height_ratios=[1, 1.1],
                              hspace=0.16, wspace=0.025)
gs_std = outer[1].subgridspec(1, 2, width_ratios=[1, 0.022], wspace=0.025)
gs_bot = outer[2].subgridspec(1, 4, width_ratios=[1, 1, 1, 0.035], wspace=0.06)

ax_top = fig.add_subplot(gs_top[0, 0]); cax_top = fig.add_subplot(gs_top[0, 1])
ax_mid = fig.add_subplot(gs_top[1, 0], sharex=ax_top); cax_mid = fig.add_subplot(gs_top[1, 1])
ax_std = fig.add_subplot(gs_std[0, 0])
map_axes = [fig.add_subplot(gs_bot[0, k], projection=ccrs.PlateCarree()) for k in (0, 1, 2)]
cax_bottom = fig.add_subplot(gs_bot[0, 3])


# --- (a) ensemble plume (same inits as panel b: capped at max_init_b) ---
plume_colors = ncp.pal_sample("Osmium", len(plume_inits))
for i, init_date in enumerate(plume_inits):
    fcst = z500_mean.sel({init_dim: init_date})
    fcst_time = pd.to_datetime(valid_time.sel({init_dim: init_date}).values)
    color = plume_colors[i]
    for member in fcst[member_dim].values:
        ax_top.plot(fcst_time, fcst.sel({member_dim: member}).values,
                    color=color, alpha=0.22, linewidth=0.6)
    ax_top.plot(fcst_time, fcst.mean(member_dim, skipna=True).values, color=color, linewidth=1.8)

era5_line, = ax_top.plot(era5_daily.index, era5_daily.values, color="black",
                         linestyle="--", linewidth=2.0, zorder=15, label="ERA5(T)")
threshold_line = ax_top.axhline(threshold, color="0.4", linewidth=1.2, linestyle=":",
                                label=f"{threshold:.0f} m threshold")
ax_top.set_ylabel(r"Box-mean $Z_{500}$ [m]")
ax_top.set_xlim(plot_start, plot_end)
set_date_axis(ax_top, plot_start, plot_end)
ax_top.tick_params(labelbottom=False)            # shared with (b)
ax_top.grid(True, alpha=0.25)
ax_top.legend(handles=[era5_line, threshold_line], loc="lower right", frameon=False)
add_panel_label(ax_top, "a")

plume_cmap = mcolors.ListedColormap(plume_colors)
plume_norm = mcolors.BoundaryNorm(np.arange(len(plume_inits) + 1) - 0.5, plume_cmap.N)
cbar_top = fig.colorbar(plt.cm.ScalarMappable(norm=plume_norm, cmap=plume_cmap), cax=cax_top)
plume_tick_step = 1 if len(plume_inits) <= 6 else 2
cbar_top.set_ticks(np.arange(0, len(plume_inits), plume_tick_step))
cbar_top.set_ticklabels(plume_inits[::plume_tick_step].strftime("%-d %b"))
cbar_top.set_label("Initialization")


# --- (b) change in exceedance probability ---
pcm_mid = ax_mid.pcolormesh(dprob_x, dprob_y, dprob_wide.values, shading="auto",
                            cmap=diverging_cmap, norm=dprob_norm)
ax_mid.axhline(mdates.date2num(highlight_init_date), color="black", linewidth=2.0, zorder=20)
ax_mid.set_ylabel("Initialization")
ax_mid.set_xlabel("Valid date")
init_tick_step = 1 if len(dprob_y_dates) <= 8 else 2
ax_mid.set_yticks(mdates.date2num(dprob_y_dates[::init_tick_step].to_pydatetime()))
ax_mid.set_yticklabels(dprob_y_dates[::init_tick_step].strftime("%-d %b"))
ax_mid.set_xlim(plot_start, plot_end)
ax_mid.grid(True, alpha=0.25, color="white", linewidth=0.6, zorder=25)
add_panel_label(ax_mid, "b")
cbar_mid = fig.colorbar(pcm_mid, cax=cax_mid)
cbar_mid.set_label("Δ probability [pp]")


# --- (c) across-member std + exceedance fraction for the target valid day ---
frac_color = "#c1121f"
ax_std.plot(std_x_inits, std_values, color="black", linewidth=1.6, marker="o", markersize=4)
ax_std.set_ylabel("Ensemble std [m]")
ax_std.set_xlabel("Initialization")
ax_std.grid(True, alpha=0.25)
ax_frac = ax_std.twinx()
ax_frac.plot(std_x_inits, frac_values, color=frac_color, linewidth=1.6,
             linestyle="--", marker="s", markersize=4)
ax_frac.set_ylabel(f"Fraction > {threshold:.0f} m", color=frac_color)
ax_frac.tick_params(axis="y", labelcolor=frac_color, colors=frac_color)
ax_frac.spines["right"].set_color(frac_color)
ax_frac.set_ylim(-0.02, 1.02)
if len(std_x_inits) > 0:
    set_date_axis(ax_std, std_x_inits.min(), std_x_inits.max(),
                  label_every=1 if len(std_x_inits) <= 14 else 2)
ax_std.set_title(" ")#f"Box-mean z500 valid {target_valid_date:%-d %b}", loc="right")
add_panel_label(ax_std, "c")


# --- (d)-(f) ensemble-mean daily-mean z500 maps for the target valid day ---
cf_bottom = None
for i, (ax, z_map, init_date, panel_label) in enumerate(
        zip(map_axes, maps, init_dates_to_plot, ["d", "e", "f"])):
    ax.set_extent([map_lon_min, map_lon_max, map_lat_min, map_lat_max], crs=ccrs.PlateCarree())
    cf_bottom = ax.contourf(z_map[lon_name], z_map[lat_name], z_map, levels=map_fill_levels,
                            cmap=sequential_cmap, extend="both", transform=ccrs.PlateCarree())
    cs = ax.contour(z_map[lon_name], z_map[lat_name], z_map, levels=map_line_levels,
                    colors="black", linewidths=0.4, alpha=0.6, transform=ccrs.PlateCarree())
    ax.clabel(cs, cs.levels[::2], inline=True, fontsize=7, fmt="%.0f")
    ax.coastlines(resolution="50m", linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3)
    ax.plot([box_lon_min, box_lon_max, box_lon_max, box_lon_min, box_lon_min],
            [box_lat_min, box_lat_min, box_lat_max, box_lat_max, box_lat_min],
            color="red", linewidth=1.5, transform=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linewidth=0.4, color="gray", alpha=0.5, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = (i == 0)
    gl.xlocator = mticker.FixedLocator([-20, 0, 20])
    gl.ylocator = mticker.FixedLocator([30, 45, 60])
    gl.xlabel_style = {"size": 8}
    gl.ylabel_style = {"size": 8}
    ax.set_title(f"Init {init_date:%-d %b}", loc="right")
    add_panel_label(ax, panel_label, inset=True)

cbar_bottom = fig.colorbar(cf_bottom, cax=cax_bottom)
cbar_bottom.set_ticks(map_line_levels[::2] if len(map_line_levels) > 8 else map_line_levels)
cbar_bottom.set_label(r"$Z_{500}$ [m]")

os.makedirs("paper_figures", exist_ok=True)
fig.savefig("paper_figures/500_combined_overview_dailymean.png", dpi=300, bbox_inches="tight")
fig.savefig("paper_figures/500_combined_overview_dailymean.pdf", bbox_inches="tight")
plt.show()